In [ ]:
from langchain_cohere import ChatCohere, CohereEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain.retrievers.document_compressors import LLMChainExtactor  # type: ignore
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever # type: ignore

In [2]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]
     


In [3]:
# create faiss vector store from the documents
embedding_model = CohereEmbeddings(model  = "embed-english-v3.0")
vectorstore = FAISS.from_documents(docs, embedding_model)

In [4]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k":5})

In [ ]:
#set up the compressor using as LLM
llm = ChatCohere(model="command-a-03-2025")
compressor = LLMChainExtactor.from_llm(llm)

In [ ]:
# create a contextual compression retriever

compression_retriever = ContextualCompressionRetriever(
    base_retriever = base_retriever,
    base_compressor = compressor
)

In [ ]:
# query the retriever
query = "What is the photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [ ]:
for i , doc in enumerate(compressed_results):
    print(f"\n---Result{i+1}---")
    print(doc.page_content)
